<a href="https://colab.research.google.com/github/ShilpaShyamal/Bankmarketingproject/blob/main/SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================
# INSTALL (run once)
# ================================
!pip install xgboost lightgbm catboost imbalanced-learn

# ================================
# IMPORT LIBRARIES
# ================================
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.metrics import geometric_mean_score

# ================================
# SAMPLE DATA (replace with your dataset)
# ================================
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# ================================
# K-FOLD = 10
# ================================
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# ================================
# METRICS FUNCTION
# ================================
def calculate_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    specificity = tn / (tn + fp)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": specificity,
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "G-Mean": geometric_mean_score(y_true, y_pred),
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ================================
# MODELS + PARAMS
# ================================
models = {
    "Logistic Regression": (LogisticRegression(), {"C":[0.1,1,10]}),

    "Decision Tree": (DecisionTreeClassifier(), {"max_depth":[3,5,10]}),

    "Random Forest": (RandomForestClassifier(), {"n_estimators":[50,100]}),

    "Gradient Boosting": (GradientBoostingClassifier(), {"n_estimators":[50,100]}),

    "XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='logloss'), {"n_estimators":[50,100]}),

    "LightGBM": (LGBMClassifier(), {"n_estimators":[50,100]}),

    "CatBoost": (CatBoostClassifier(verbose=0), {"iterations":[50,100]}),

    "SVM": (SVC(probability=True), {"C":[0.1,1]}),

    "KNN": (KNeighborsClassifier(), {"n_neighbors":[3,5,7]}),

    "MLP": (MLPClassifier(max_iter=500), {"hidden_layer_sizes":[(50,), (100,)]}),

    "Bagging": (BaggingClassifier(), {"n_estimators":[10,50]}),

    "Stacking": (StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier()),
            ('dt', DecisionTreeClassifier())
        ],
        final_estimator=LogisticRegression()
    ), {"final_estimator__C":[0.1,1]})
}

# ================================
# MAIN LOOP
# ================================
all_results = []

for name, (model, params) in models.items():
    print(f"\n==================== {name} ====================")

    search = RandomizedSearchCV(
        model,
        param_distributions=params,
        n_iter=3,
        cv=10,   # 🔥 10-fold
        scoring='accuracy',
        n_jobs=-1,
        random_state=42
    )

    search.fit(X, y)

    best_model = search.best_estimator_

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y), 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        best_model.fit(X_train, y_train)

        y_pred = best_model.predict(X_test)

        if hasattr(best_model, "predict_proba"):
            y_prob = best_model.predict_proba(X_test)[:,1]
        else:
            y_prob = y_pred

        metrics = calculate_metrics(y_test, y_pred, y_prob)
        fold_results.append(metrics)

        print(f"Fold {fold}:", metrics)

    df = pd.DataFrame(fold_results)
    mean_result = df.mean()

    print("\nAverage Result:")
    print(mean_result)

    mean_result["Model"] = name
    all_results.append(mean_result)

# ================================
# FINAL RESULT TABLE
# ================================
final_df = pd.DataFrame(all_results)
final_df = final_df.set_index("Model")

print("\n================ FINAL RESULT ================")
print(final_df)

# SAVE CSV
final_df.to_csv("/content/bank-additional-full (1) (1) (1).csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00

==================== Logistic Regression ====================
Fold 1: {'Accuracy': 0.84, 'Precision': 0.8863636363636364, 'Recall': 0.78, 'Specificity': np.float64(0.9), 'F1': 0.8297872340425532, 'ROC-AUC': np.float64(0.9119999999999999), 'MCC': np.float64(0.6849495194215732), 'G-Mean': np.float64(0.8378544026261365), 'Kappa': np.float64(0.6799999999999999)}
Fold 2: {'Accuracy': 0.91, 'Precision': 0.9183673469387755, 'Recall': 0.9, 'Specificity': np.float64(0.92), 'F1': 0.9090909090909091, 'ROC-AUC': np.float64(0.942), 'MCC': np.float64(0.8201640492164058), 'G-Mean': np.float64(0.9099450532861861), 'Kappa': np.float64(0.8200000000000001)}
Fold 3: {'Accuracy': 0.86, 'Precision': 0.86, 'Recall': 0.86, 'Specificity': np.float64(0.86), 'F1': 0.86, 'ROC-AUC': np.float64(0.9376000000000001), 'MCC': np.float64(0.72), 'G-Mean': np.float64(0.86), 'Kappa': np.float64(0.72)}
Fold 4: {'Accuracy': 0.92, 'Precision': 0.95

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fold 1: {'Accuracy': 0.88, 'Precision': 0.9130434782608695, 'Recall': 0.84, 'Specificity': np.float64(0.92), 'F1': 0.875, 'ROC-AUC': np.float64(0.9309999999999999), 'MCC': np.float64(0.7624437362098715), 'G-Mean': np.float64(0.8790904390334364), 'Kappa': np.float64(0.76)}
Fold 2: {'Accuracy': 0.9, 'Precision': 0.9, 'Recall': 0.9, 'Specificity': np.float64(0.9), 'F1': 0.9, 'ROC-AUC': np.float64(0.9384), 'MCC': np.float64(0.8), 'G-Mean': np.float64(0.9), 'Kappa': np.float64(0.8)}
Fold 3: {'Accuracy': 0.9, 'Precision': 0.8846153846153846, 'Recall': 0.92, 'Specificity': np.float64(0.88), 'F1': 0.9019607843137255, 'ROC-AUC': np.float64(0.9774), 'MCC': np.float64(0.8006407690254356), 'G-Mean': np.float64(0.8997777503361595), 'Kappa': np.float64(0.8)}
Fold 4: {'Accuracy': 0.92, 'Precision': 0.9565217391304348, 'Recall': 0.88, 'Specificity': np.float64(0.96), 'F1': 0.9166666666666666, 'ROC-AUC': np.float64(0.968), 'MCC': np.float64(0.8427009716003844), 'G-Mean': np.float64(0.9191300234460845),

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fold 1: {'Accuracy': 0.86, 'Precision': 0.8913043478260869, 'Recall': 0.82, 'Specificity': np.float64(0.9), 'F1': 0.8541666666666666, 'ROC-AUC': np.float64(0.9308), 'MCC': np.float64(0.7223151185146152), 'G-Mean': np.float64(0.8590692637965812), 'Kappa': np.float64(0.72)}
Fold 2: {'Accuracy': 0.89, 'Precision': 0.8979591836734694, 'Recall': 0.88, 'Specificity': np.float64(0.9), 'F1': 0.8888888888888888, 'ROC-AUC': np.float64(0.9420000000000001), 'MCC': np.float64(0.7801560468156056), 'G-Mean': np.float64(0.8899438184514795), 'Kappa': np.float64(0.78)}
Fold 3: {'Accuracy': 0.92, 'Precision': 0.92, 'Recall': 0.92, 'Specificity': np.float64(0.92), 'F1': 0.92, 'ROC-AUC': np.float64(0.9576), 'MCC': np.float64(0.84), 'G-Mean': np.float64(0.92), 'Kappa': np.float64(0.84)}
Fold 4: {'Accuracy': 0.94, 'Precision': 0.9782608695652174, 'Recall': 0.9, 'Specificity': np.float64(0.98), 'F1': 0.9375, 'ROC-AUC': np.float64(0.9675999999999999), 'MCC': np.float64(0.8828295892956407), 'G-Mean': np.float64

...

Fold 9: {'Accuracy': 0.89, 'Precision': 0.8545454545454545, 'Recall': 0.94, 'Specificity': np.float64(0.84), 'F1': 0.8952380952380953, 'ROC-AUC': np.float64(0.9596), 'MCC': np.float64(0.7839294959021854), 'G-Mean': np.float64(0.8885943956609224), 'Kappa': np.float64(0.78)}


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fold 10: {'Accuracy': 0.73, 'Precision': 0.7804878048780488, 'Recall': 0.64, 'Specificity': np.float64(0.82), 'F1': 0.7032967032967034, 'ROC-AUC': np.float64(0.8755999999999999), 'MCC': np.float64(0.46763811563329066), 'G-Mean': np.float64(0.7244308110509934), 'Kappa': np.float64(0.45999999999999996)}

Average Result:
Accuracy       0.829000
Precision      0.829120
Recall         0.830000
Specificity    0.828000
F1             0.827419
ROC-AUC        0.919560
MCC            0.661320
G-Mean         0.827365
Kappa          0.658000
dtype: float64

==================== Bagging ====================
Fold 1: {'Accuracy': 0.9, 'Precision': 0.9347826086956522, 'Recall': 0.86, 'Specificity': np.float64(0.94), 'F1': 0.8958333333333334, 'ROC-AUC': np.float64(0.9212), 'MCC': np.float64(0.8025723539051279), 'G-Mean': np.float64(0.8991106717195608), 'Kappa': np.float64(0.8)}
Fold 2: {'Accuracy': 0.92, 'Precision': 0.9375, 'Recall': 0.9, 'Specificity': np.float64(0.94), 'F1': 0.9183673469387755, 'ROC

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=3. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fold 1: {'Accuracy': 0.86, 'Precision': 0.875, 'Recall': 0.84, 'Specificity': np.float64(0.88), 'F1': 0.8571428571428571, 'ROC-AUC': np.float64(0.9192), 'MCC': np.float64(0.7205766921228921), 'G-Mean': np.float64(0.859767410408187), 'Kappa': np.float64(0.72)}
Fold 2: {'Accuracy': 0.9, 'Precision': 0.9, 'Recall': 0.9, 'Specificity': np.float64(0.9), 'F1': 0.9, 'ROC-AUC': np.float64(0.9368), 'MCC': np.float64(0.8), 'G-Mean': np.float64(0.9), 'Kappa': np.float64(0.8)}
Fold 3: {'Accuracy': 0.91, 'Precision': 0.9019607843137255, 'Recall': 0.92, 'Specificity': np.float64(0.9), 'F1': 0.9108910891089109, 'ROC-AUC': np.float64(0.9726), 'MCC': np.float64(0.8201640492164058), 'G-Mean': np.float64(0.9099450532861861), 'Kappa': np.float64(0.8200000000000001)}
Fold 4: {'Accuracy': 0.94, 'Precision': 0.9782608695652174, 'Recall': 0.9, 'Specificity': np.float64(0.98), 'F1': 0.9375, 'ROC-AUC': np.float64(0.967), 'MCC': np.float64(0.8828295892956407), 'G-Mean': np.float64(0.9391485505499116), 'Kappa': n